# 04 — Feature Engineering

Three jobs in this notebook:

1. **Build position-invariant aggregate features** (mean/std/min/max over
   whatever months are valid, plus valid-month counts) for both the
   masked-augmented train and real test, using `src/features.py`.
2. **Run the adversarial-validation check we deferred from notebook 02.**
   Now that train (augmented) and test have matching window-composition
   (verified in notebook 03) and matching aggregate feature representations,
   we can finally ask the real question: once you control for *how much* of
   the year each row sees, do train and test still look different? Notebook
   02's raw KS-comparison couldn't answer this cleanly because it mixed
   genuine drift with the masking-composition confound.
3. **Decide the feature set** going into notebook 05's model, informed by
   what adversarial validation actually shows rather than domain assumptions
   alone.

Output: `data/processed/train_features.csv`, `data/processed/test_features.csv`.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

from src.config import ALL_BANDS, MONTHS, PROCESSED_DIR, RANDOM_SEED
from src.features import (
    compute_month_indices, build_aggregate_feature_table,
    aggregate_feature_columns, DERIVED_INDEX_NAMES,
)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 30)

train_aug = pd.read_csv(PROCESSED_DIR / 'train_augmented.csv')
test_clean = pd.read_csv(PROCESSED_DIR / 'test_clean.csv')
print(train_aug.shape, test_clean.shape)

(18210, 150) (1030, 145)


## 1. Sanity check: do S2 bands share the same per-month missingness?

In [2]:
# If cloud cover masks a whole optical acquisition at once, every S2 band
# should be missing together for a given (row, month) -- not independently.
# Worth confirming on REAL test before we rely on a single indicator band.
s2_bands_check = ['blue', 'green', 'red', 'nir', 'swir1']
mismatch_count = 0
for m in MONTHS:
    cols = [f'{b}_{m}' for b in s2_bands_check]
    missing_pattern = test_clean[cols].isna()
    # rows where not all S2 bands agree on missing/present
    disagreement = missing_pattern.nunique(axis=1) > 1
    mismatch_count += disagreement.sum()

print(f"(row, month) pairs where S2 bands disagree on missingness: {mismatch_count}")
print(f"out of {len(test_clean) * len(MONTHS)} total (row, month) pairs")

(row, month) pairs where S2 bands disagree on missingness: 0
out of 12360 total (row, month) pairs


**Why this matters:** if this count is ~0, it confirms S2 bands are masked
together per month (as expected physically — one cloud obscures the whole
optical scene, not just one band), which justifies using a single band
(`blue`) as the S2-presence indicator elsewhere in the codebase, rather than
checking all 9 optical bands individually every time.

## 2. Compute per-month indices, then collapse to aggregate feature tables

In [3]:
train_idx = compute_month_indices(train_aug)
test_idx = compute_month_indices(test_clean)

train_features = build_aggregate_feature_table(train_idx)
test_features = build_aggregate_feature_table(test_idx)

feature_cols = aggregate_feature_columns(train_features)
print(f"{len(feature_cols)} model-input features per row")
print(train_features.shape, test_features.shape)
train_features.head()

67 model-input features per row
(18210, 70) (1030, 68)


,ID,label,origin_id,VH_mean,VH_std,VH_min,VH_max,VV_mean,VV_std,VV_min,VV_max,blue_mean,blue_std,blue_min,blue_max,...,mndwi_mean,mndwi_std,mndwi_min,mndwi_max,ndvi_mean,ndvi_std,ndvi_min,ndvi_max,vh_minus_vv_mean,vh_minus_vv_std,vh_minus_vv_min,vh_minus_vv_max,n_valid_s1_months,n_valid_s2_months,s2_valid_fraction_of_s1
0,ID_TR_NEW_XVGKFMLNRJ_v0,0,ID_TR_NEW_XVGKFMLNRJ,-29.567864,3.701085,-36.058930,-25.347438,-22.032904,1.216922,-23.955628,-20.290513,1868.166667,200.921295,1659.0,2187.0,...,0.232590,0.051984,0.165379,0.287002,-0.099785,0.069322,-0.186883,-0.016949,-7.534959,3.373430,-14.197400,-5.056925,6,6,1.0
1,ID_TR_NEW_GP8KNSWVP6_v0,0,ID_TR_NEW_GP8KNSWVP6,-17.757554,2.277168,-20.684984,-15.908768,-9.086967,1.783721,-10.697351,-6.585548,1781.750000,306.112588,1381.0,2045.0,...,-0.216348,0.040738,-0.266813,-0.174325,0.288448,0.267322,0.062903,0.641135,-8.670586,1.407952,-9.987633,-6.762008,4,4,1.0
2,ID_TR_NEW_87X3957MVS_v0,1,ID_TR_NEW_87X3957MVS,-28.749292,2.188349,-32.687598,-26.606786,-20.630447,1.152897,-22.524134,-19.338749,1645.500000,183.308210,1437.0,1906.0,...,0.141804,0.044079,0.055138,0.173913,-0.042948,0.034959,-0.100798,0.000825,-8.118845,2.853379,-13.348849,-6.089281,6,6,1.0
3,ID_TR_NEW_T4JMRPKHS3_v0,0,ID_TR_NEW_T4JMRPKHS3,-16.571281,0.995688,-17.744163,-15.318855,-6.095938,2.020954,-8.183441,-3.566426,1299.200000,82.823306,1184.0,1391.0,...,-0.152717,0.031046,-0.190288,-0.118561,0.276705,0.143948,0.059753,0.405024,-10.475343,2.306584,-13.767643,-7.562522,5,5,1.0
4,ID_TR_NEW_2CTUQQ8KLU_v0,0,ID_TR_NEW_2CTUQQ8KLU,-28.479219,1.190637,-30.122815,-27.595630,-23.296417,2.250394,-26.377800,-21.313399,1939.750000,211.326564,1734.0,2164.0,...,0.202966,0.062492,0.140261,0.280740,-0.123327,0.072796,-0.192515,-0.048861,-5.182802,2.639344,-6.613239,-1.230547,4,4,1.0


## 3. Save feature tables

In [4]:
train_features.to_csv(PROCESSED_DIR / 'train_features.csv', index=False)
test_features.to_csv(PROCESSED_DIR / 'test_features.csv', index=False)
print("saved.")

saved.


## 4. Adversarial validation (the real version, deferred from notebook 02)

Combine train_features (label removed) and test_features, add a binary
`is_test` target, and see whether a classifier can tell them apart using
these matched aggregate features. Because `train_features` still has 10
masked variants per original location (`origin_id`), we group-fold on
`origin_id` for the train side (test rows are already unique, so their own
`ID` works as a group) — otherwise the classifier could "cheat" by
recognizing a specific location's fingerprint across variants rather than
learning genuine train/test distributional shift.

In [5]:
adv_train = train_features[feature_cols].copy()
adv_train['is_test'] = 0
adv_train['group'] = train_features['origin_id']

adv_test = test_features[feature_cols].copy()
adv_test['is_test'] = 1
adv_test['group'] = test_features['ID']  # unique per row, no grouping needed here

adv_data = pd.concat([adv_train, adv_test], ignore_index=True)
print(adv_data['is_test'].value_counts())

gkf = GroupKFold(n_splits=5)
fold_aucs = []
importances = np.zeros(len(feature_cols))

X = adv_data[feature_cols]
y = adv_data['is_test']
groups = adv_data['group']

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    model = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        is_unbalance=True, random_state=RANDOM_SEED, verbosity=-1,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    preds = model.predict_proba(X.iloc[val_idx])[:, 1]
    auc = roc_auc_score(y.iloc[val_idx], preds)
    fold_aucs.append(auc)
    importances += model.feature_importances_ / gkf.n_splits
    print(f"fold {fold}: AUC = {auc:.4f}")

print()
print(f"mean adversarial-validation AUC: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")

is_test
0    18210
1     1030
Name: count, dtype: int64


fold 0: AUC = 0.9794


fold 1: AUC = 0.9807


fold 2: AUC = 0.9896


fold 3: AUC = 0.9911


fold 4: AUC = 0.9794

mean adversarial-validation AUC: 0.9841 (+/- 0.0052)


**How to read this AUC:**
- **~0.5** means the classifier can't tell train from test at all — our
  aggregate features are, for practical purposes, from the same distribution.
  Great news: standard CV should generalize well to the leaderboard.
- **~0.6-0.75** means there's detectable but moderate shift — worth checking
  which features are driving it (next cell) and possibly downweighting them.
- **~0.9+** means train and test are highly separable even after all our
  matching work — genuine cross-period drift is strong, and we should treat
  CV scores in notebook 05 with real skepticism, leaning more on whatever
  features rank low in the importance list below.

In [6]:
adv_importance = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
adv_importance.head(20)

vh_minus_vv_mean    209.2
mndwi_min           195.6
mndwi_mean          175.8
VH_min              140.0
ndvi_max            137.8
ndwi_min            134.0
VH_std              132.8
blue_mean           126.0
VV_min              125.8
vh_minus_vv_max     123.8
mndwi_std           123.0
ndwi_max            121.8
mndwi_max           119.0
swir1_min           117.2
VV_mean             117.2
swir2_std           114.6
ndvi_min            111.0
VV_max              109.8
blue_max            106.4
vh_minus_vv_min      99.6
dtype: float64

**How to read this:** these are the features the adversarial classifier
relied on most to distinguish train from test. High-importance features here
are the ones most likely to encode train/test drift rather than genuine
pond/non-pond signal — worth cross-referencing against notebook 02's
separability ranking (features that are high on *both* lists are a real
tension: useful for the actual task, but also a shift risk).

## 5. Cross-reference: separability (from notebook 02) vs. adversarial importance (here)

In [7]:
from scipy import stats

# recompute the same univariate separability score as notebook 02, but now
# on the (masked-augmented) mean-aggregate features, for a fair side-by-side
sep_rows = []
for feat in ALL_BANDS + DERIVED_INDEX_NAMES:
    col = f'{feat}_mean'
    valid = train_features[col].notna()
    corr, _ = stats.pointbiserialr(train_features.loc[valid, 'label'], train_features.loc[valid, col])
    sep_rows.append({'feature': feat, 'abs_label_corr': abs(corr)})
sep_df = pd.DataFrame(sep_rows).set_index('feature')

adv_mean_only = adv_importance[[f'{f}_mean' for f in ALL_BANDS + DERIVED_INDEX_NAMES]]
adv_mean_only.index = [i.replace('_mean', '') for i in adv_mean_only.index]

comparison = sep_df.join(adv_mean_only.rename('adversarial_importance'))
comparison.sort_values('abs_label_corr', ascending=False)

,abs_label_corr,adversarial_importance
feature,,
ndwi,0.702065,75.8
mndwi,0.693044,175.8
ndvi,0.630139,78.8
nira,0.584426,28.6
VH,0.583388,94.0
nir,0.567735,73.0
swir1,0.556988,79.4
re3,0.554014,15.4
re2,0.546965,24.6


## Summary of findings (actual results)

- S2 band missingness agreement check: **0 disagreements** out of 12,360
  (row, month) pairs — confirms cloud masking hits all optical bands together,
  justifying `blue` as a single S2-presence indicator throughout the codebase.
- Built **67 aggregate features** per location (16 band/index groups x
  mean/std/min/max, plus 3 valid-month-count features).
- **Adversarial-validation AUC: 0.984** (full augmented set, grouped by
  origin) — and **0.980** on the declumped singleton check (one variant per
  location, ruling out the augmentation-clustering confound). These are
  close, which means the high separability is NOT an artifact of our
  augmentation creating clustered near-duplicates — it reflects **genuine,
  substantial distributional drift** between the train and test time periods,
  exactly matching the competition's explicit warning. This is the strongest,
  best-controlled signal we have on how serious that drift is.
- Nearly every feature contributes to this separability (top adversarial
  importances spread across `vh_minus_vv_mean`, `mndwi_min/mean`, `VH_min`,
  `ndvi_max`, `ndwi_min`, `VH_std`, `blue_mean`, `VV_min`, and more) — this
  isn't one leaky feature to drop, it's pervasive.
- Cross-referencing separability (task signal) vs. adversarial importance
  (shift signal) per band/index (§5):
  - **Best signal-to-shift ratio** (good for the task, low shift risk):
    `re3` (corr 0.55 / adv 15.4), `re2` (0.55 / 24.6), `nira` (0.58 / 28.6).
  - **High value but high shift risk** (keep, but treat cautiously):
    `mndwi` (0.69 / 175.8), `VH` (0.58 / 94.0), `VV` (0.49 / 117.2),
    `vh_minus_vv` (0.46 / 209.2 — highest shift driver of all).
  - **Poor trade-off** (weak task signal, meaningful shift signal):
    `blue` (0.18 / 126.0), `red` (0.04 / 71.4) — the raw aggregates of these
    are strong drop candidates; their contribution to the water/vegetation
    indices is preserved separately, so dropping the raw columns doesn't
    remove that signal, just the riskier direct copy of it.

**Carried into notebook 05 — this changes the modeling strategy, not just the
feature list:**
1. **Don't trust plain CV.** With AUC ~0.98 separability, a GroupKFold score
   on augmented train will likely overstate real leaderboard performance,
   because validation folds are still drawn from the same train-period
   distribution as the training folds. Expect a gap, and don't chase CV score
   in isolation.
2. **Consider importance-weighting training rows** by the adversarial model's
   predicted P(is_test) — a standard, lightweight domain-adaptation trick:
   training rows that "look more like test" get upweighted, nudging the
   model toward the region of feature space it'll actually be evaluated on.
3. **Favor simpler/more regularized models over aggressively tuned ones.** A
   model that fits train-period quirks tightly is fitting noise that won't
   transfer; shallower trees, stronger regularization, or fewer high-shift
   features are more likely to generalize than squeezing out the last bit of
   CV score.
4. **Drop or deprioritize `blue`/`red` raw aggregates** given their poor
   signal-to-shift trade-off; lean on `re2`/`re3`/`nira` and the water indices
   as the core signal, while keeping `mndwi`/`VH`/`VV`/`vh_minus_vv` in play
   despite their shift risk since they're too predictive to discard outright.

## 6. Confound check: is the 0.98 AUC genuine drift, or an artifact of augmentation clustering?

`train_features` currently has 10 near-duplicate variants per original
location (correlated aggregate stats, since they're subsampled windows of the
same underlying 12 months), while every test row is an independent, unique
location. That structural difference — clumped vs. singleton — could inflate
adversarial AUC on its own, with nothing to do with genuine train/test period
drift. `GroupKFold` prevents the classifier from literally *memorizing* a
specific location's identity across folds, but it does NOT remove this
different-clustering-shape effect, since that's a property of the whole
feature-space distribution, not of any one leaked example.

To isolate genuine drift from this clustering artifact, rerun the exact same
check using only ONE variant per original location (so both sides are
independent singletons).

In [8]:
singleton_mask = train_features['ID'].str.endswith('_v0')
train_features_singleton = train_features[singleton_mask].reset_index(drop=True)
print("singleton train rows:", len(train_features_singleton), "(should equal original 1,821 train rows)")

adv_train_single = train_features_singleton[feature_cols].copy()
adv_train_single['is_test'] = 0
adv_train_single['group'] = train_features_singleton['origin_id']

adv_data_single = pd.concat([adv_train_single, adv_test], ignore_index=True)
print(adv_data_single['is_test'].value_counts())

X2 = adv_data_single[feature_cols]
y2 = adv_data_single['is_test']
groups2 = adv_data_single['group']

fold_aucs_single = []
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X2, y2, groups2)):
    model = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        is_unbalance=True, random_state=RANDOM_SEED, verbosity=-1,
    )
    model.fit(X2.iloc[tr_idx], y2.iloc[tr_idx])
    preds = model.predict_proba(X2.iloc[val_idx])[:, 1]
    auc = roc_auc_score(y2.iloc[val_idx], preds)
    fold_aucs_single.append(auc)
    print(f"fold {fold}: AUC = {auc:.4f}")

print()
print(f"mean adversarial-validation AUC (singleton, declumped): {np.mean(fold_aucs_single):.4f} (+/- {np.std(fold_aucs_single):.4f})")
print(f"(for comparison, full-augmented-set AUC was {np.mean(fold_aucs):.4f})")

singleton train rows: 1821 (should equal original 1,821 train rows)
is_test
0    1821
1    1030
Name: count, dtype: int64


fold 0: AUC = 0.9733


fold 1: AUC = 0.9780


fold 2: AUC = 0.9885


fold 3: AUC = 0.9766


fold 4: AUC = 0.9827

mean adversarial-validation AUC (singleton, declumped): 0.9798 (+/- 0.0053)
(for comparison, full-augmented-set AUC was 0.9841)


**Interpreting the comparison:** if the singleton AUC comes back close to the
full-set AUC, the clustering artifact wasn't the explanation — the shift is
genuinely in the values themselves, consistent with the competition's explicit
warning about training and test periods differing. If the singleton AUC drops
substantially toward 0.5, the clustering structure was doing a lot of the
earlier work, and the real drift is milder than 0.98 suggested.